# Entorno de Colab — canario de la vía de entrenamiento

**Política del proyecto:** esta PC **no entrena** modelos de IA. Todo entrenamiento se
hace en Google Colab, y el framework es **TensorFlow**. Este notebook es el canario:
no entrena nada, verifica que la sesión está sana. Ejecútalo **antes** de lanzar un
entrenamiento real; si alguna celda falla, la sesión no sirve.

Motivo de la política (medido en la máquina local, ver `Sessions/2026-09-26_capacidad_hardware_pc_ml.md`):
Celeron N4020 de 2 núcleos a 1,1 GHz, 3,63 GB de RAM, sin GPU utilizable. TensorFlow sí
carga ahí (las wheels modernas hacen *dispatch* en runtime y se adaptan a SSE4.2, sin
AVX), así que no es una imposibilidad técnica: es una decisión de política.

## 1. Permistencia — lo primero, y no es opcional

Las sesiones de Colab son **efímeras**: al cerrarlas se pierde todo lo que no esté en
Drive. Este es el punto ciego clásico de trabajar en Colab, así que va primero. Si el
Drive no monta, **no sigas**: cualquier modelo entrenado en esta sesión desaparecería.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Convention del proyecto: todo artefacto (datasets, checkpoints, modelos) vive en Drive,
# nunca en /content, que se borra al cerrar la sesion. Cambiar esta raiz si mas adelante
# se adopta otra convencion.
RAIZ = Path('/content/drive/MyDrive/electronica_ml')
for sub in ('datasets', 'checkpoints', 'modelos', 'resultados'):
    (RAIZ / sub).mkdir(parents=True, exist_ok=True)

print("Drive montado. Raiz de artefactos:", RAIZ)
print("Persistente :", RAIZ.exists())
print("Efimero     : /content  (se pierde al cerrar la sesion)")

## 2. Inventario de la sesion

In [ ]:
import sys, platform, os, shutil

try:
    import psutil
    vm = psutil.virtual_memory()
    ram_tot, ram_disp = vm.total / 1e9, vm.available / 1e9
except ImportError:
    ram_tot = ram_disp = float('nan')

print(f"Python : {sys.version.split()[0]}")
print(f"SO     : {platform.system()} {platform.release()}")
print(f"RAM    : {ram_tot:.1f} GB totales, {ram_disp:.1f} GB disponibles")
print(f"CPU    : {os.cpu_count()} nucleos logicos")
print(f"Disco  : {shutil.disk_usage('/content')[2] / 1e9:.1f} GB libres en /content")

## 3. TensorFlow y GPU — el check clave

In [ ]:
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detectadas: {len(gpus)}")
for g in gpus:
    detalles = tf.config.experimental.get_device_details(g)
    print(f"  {g.name}")
    for clave, valor in detalles.items():
        print(f"    {clave}: {valor}")

## 4. Prueba de computo real

Detectar la GPU no es lo mismo que **computar** en ella: un runtime puede anunciar la GPU
y fallar al usarla. Esta celda fuerza la operacion en GPU y cae a CPU si no puede.

In [ ]:
import time

N = 2000
if gpus:
    with tf.device('/GPU:0'):
        a = tf.random.normal((N, N))
        b = tf.random.normal((N, N))
        t0 = time.time()
        c = tf.matmul(a, b)
        _ = c.numpy()          # forzamos la ejecucion: sin esto es lazy
        dt_gpu = time.time() - t0
    print(f"matmul {N}x{N} en GPU: {dt_gpu:.3f} s")
    print("RESULTADO: GPU OK")
else:
    print("RESULTADO: sin GPU, solo CPU. Valido para depurar, no para entrenar.")

# Comparativa CPU, para tener referencia de la brecha
a = tf.random.normal((N, N)); b = tf.random.normal((N, N))
t0 = time.time(); _ = tf.matmul(a, b).numpy(); dt_cpu = time.time() - t0
print(f"matmul {N}x{N} en CPU : {dt_cpu:.3f} s")
if gpus and dt_gpu > 0:
    print(f"Aceleracion          : {dt_cpu / dt_gpu:.1f}x")

## 5. Criterio de veredicto

La sesion sirve para entrenar si se cumple **todo** esto:

| Celda | Debe cumplirse |
|---|---|
| 1. Drive | `Persistente: True` — sin esto se pierde el trabajo |
| 3. TF y GPU | `GPUs detectadas: >= 1` |
| 4. Computo | `RESULTADO: GPU OK` y aceleracion > 1x |

Si falla la 1, monta Drive antes de seguir. Si falla la 3 o la 4, la sesion sirve para
depurgar codigo pero no para entrenar: pide un runtime con GPU accelerated.